<a href="https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [8]:
!git clone https://github.com/Ahmedali3ff/Flyrank-internship-.git

Cloning into 'Flyrank-internship-'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 144 (delta 55), reused 100 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.85 MiB | 4.57 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [9]:
import os

DATA_PATH = "/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(DATA_PATH))
print("Path:", DATA_PATH)

File exists: True
Path: /content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv


### Purpose

This section turns the validated model output into a ranked content-action queue.

The queue is intended to help a human reviewer decide which content should be reviewed first and why.

Actions are recommendations for review, not automatic publishing or deletion decisions.

In [12]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Recreate target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
    ).astype(int)

# Create a simple risk score for prioritization.
# Higher score = stronger reason to review the content.
df["priority_score"] = (
    df["content_age_days"].fillna(0) / 365
        + df["avg_position"].fillna(0) # Changed from position_score to avg_position
            + (df["impressions_last_30d"].fillna(0) == 0).astype(int)
            )

# Reason codes
def get_reason(row):
    reasons = []

    if row["content_age_days"] >= 365:
        reasons.append("STALE_CONTENT")

    if row["avg_position"] > 20:
        reasons.append("LOW_SEARCH_POSITION")

    if row["impressions_last_30d"] < row["impressions_prev_30d"]:
        reasons.append("IMPRESSION_DECLINE")
    if row["sessions_last_30d"] < row["sessions_prev_30d"]:
        reasons.append("SESSION_DECLINE")

    if row["impressions_last_30d"] == 0:
        reasons.append("NO_RECENT_IMPRESSIONS")

    if not reasons:
        reasons.append("GENERAL_REVIEW")

    return ", ".join(reasons)


df["reason_code"] = df.apply(get_reason, axis=1)

# Recommended action
def get_action(row):
    if row["content_age_days"] >= 365:
        return "Review for refresh"

    if row["impressions_last_30d"] < row["impressions_prev_30d"]:
        return "Review traffic decline"

    if row["avg_position"] > 20:
        return "Review SEO/content alignment"

    return "Human review"

df["recommended_action"] = df.apply(get_action, axis=1)

# Ranked queue
ranked_queue = (
    df.sort_values("priority_score", ascending=False)
        [
            [
                "content_id",
                "client_id",
                "priority_score",
                "reason_code",
                "recommended_action",
                "content_age_days",
                "avg_position",
                "impressions_last_30d",
                "impressions_prev_30d",
                "sessions_last_30d",
                "sessions_prev_30d",
                "is_declining_label"
            ]
        ]
        .reset_index(drop=True)
    )

ranked_queue.insert(0, "rank", ranked_queue.index + 1)

print("Ranked queue created successfully.")
print("Queue size:", len(ranked_queue))

display(ranked_queue.head(10))

Dataset shape: (30000, 44)
Ranked queue created successfully.
Queue size: 30000


,rank,content_id,client_id,priority_score,reason_code,recommended_action,content_age_days,avg_position,impressions_last_30d,impressions_prev_30d,sessions_last_30d,sessions_prev_30d,is_declining_label
0,1,content_661e1745db72,client_e29c9c180c,245.852055,"LOW_SEARCH_POSITION, SESSION_DECLINE",Review SEO/content alignment,311,245.0,1,0,0,1,0
1,2,content_23f1cc8851a9,client_e29c9c180c,184.950685,LOW_SEARCH_POSITION,Review SEO/content alignment,347,184.0,1,0,1,0,0
2,3,content_7275a6a3a8eb,client_e29c9c180c,166.130137,LOW_SEARCH_POSITION,Review SEO/content alignment,230,165.5,2,0,1,1,0
3,4,content_71a31b831092,client_e29c9c180c,161.852055,LOW_SEARCH_POSITION,Review SEO/content alignment,311,161.0,1,0,0,0,0
4,5,content_42c7c72b8391,client_e29c9c180c,146.379452,LOW_SEARCH_POSITION,Review SEO/content alignment,321,145.5,2,0,0,0,0
5,6,content_cb6c7d58c0bc,client_e29c9c180c,145.450685,LOW_SEARCH_POSITION,Review SEO/content alignment,347,144.5,2,0,1,1,0
6,7,content_692fda8c52bd,client_e29c9c180c,142.630137,"LOW_SEARCH_POSITION, SESSION_DECLINE",Review SEO/content alignment,230,142.0,1,0,0,1,0
7,8,content_3e087a5d8f15,client_e29c9c180c,139.657534,"LOW_SEARCH_POSITION, SESSION_DECLINE",Review SEO/content alignment,313,138.8,3,3,0,2,0
8,9,content_13bbd72aea33,client_e29c9c180c,119.950685,"LOW_SEARCH_POSITION, IMPRESSION_DECLINE, SESSI...",Review traffic decline,347,118.0,0,3,0,1,1
9,10,content_abeb1aa40158,client_e29c9c180c,114.453425,"LOW_SEARCH_POSITION, SESSION_DECLINE",Review SEO/content alignment,348,113.5,2,0,0,3,0


In [14]:
# Rebuild a normalized priority score
# Each signal contributes a small, interpretable amount.

df["priority_score"] = (
    (df["content_age_days"].fillna(0) / 365).clip(0, 1)
        + np.where(df["avg_position"].fillna(0) > 20, 1, 0)
            + np.where(
                    df["impressions_last_30d"].fillna(0)
                            < df["impressions_prev_30d"].fillna(0),
                                    1,
                                            0
                                                )
                                                    + np.where(
                                                            df["sessions_last_30d"].fillna(0)
                                                                    < df["sessions_prev_30d"].fillna(0),
                                                                            1,
                                                                                    0
                                                                                        )
                                                                                        )

ranked_queue = (
    df.sort_values("priority_score", ascending=False)
        [
            [
                "content_id",
                "client_id",
                "priority_score",
                "reason_code",
                "recommended_action",
                "content_age_days",
                "avg_position",
                "impressions_last_30d",
                "impressions_prev_30d",
                "sessions_last_30d",
                "sessions_prev_30d"
            ]
        ]
        .reset_index(drop=True)
    )

ranked_queue.insert(0, "rank", ranked_queue.index + 1)

print("Ranked queue rebuilt successfully.")
display(ranked_queue.head(10))

Ranked queue rebuilt successfully.


,rank,content_id,client_id,priority_score,reason_code,recommended_action,content_age_days,avg_position,impressions_last_30d,impressions_prev_30d,sessions_last_30d,sessions_prev_30d
0,1,content_07ea0872b973,client_2c624232cd,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,517,55.3,44,50,1,5
1,2,content_5c878553bcfc,client_19581e27de,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,482,27.6,83,128,1,3
2,3,content_d9878c9ba883,client_4e07408562,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,487,24.5,149,169,0,1
3,4,content_8c1e4199d4b0,client_3fdba35f04,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,460,23.5,144,218,0,1
4,5,content_c6d0030cd91a,client_19581e27de,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,480,63.2,4,13,3,9
5,6,content_4ef8621d34cd,client_e629fa6598,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,460,20.1,1,2,0,1
6,7,content_048f3a500f56,client_4e07408562,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,421,26.3,113,169,0,2
7,8,content_6f24984bd099,client_4e07408562,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,421,21.2,122,158,1,2
8,9,content_405643505f4f,client_2c624232cd,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,517,31.5,48,49,9,26
9,10,content_bf52f65fcdf4,client_19581e27de,4.0,"STALE_CONTENT, LOW_SEARCH_POSITION, IMPRESSION...",Review for refresh,389,35.5,49,77,1,3


## 2) Intended use and limits

### Intended use

This action queue is a decision-support tool for prioritizing content for human review.

It ranks content using observable signals such as content age, search position, and recent changes in impressions and sessions.

The queue helps reviewers decide which content to investigate first.

### Limits

The priority score is a heuristic ranking score, not a probability of decline.

A high-priority item does not prove that the content will decline.

The recommended action is only a starting point for human investigation.

Recommendations should be checked against the actual content, search intent, and business context before any change is made.

The queue is intended for research and prioritization, not fully automated production decisions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3) Human review and no-go list

### Human review rules

Every recommended action must be reviewed by a person before implementation.

The reviewer should:

1. Inspect the underlying content.
2. Check the relevant performance signals.
3. Verify that the recommended action matches the search intent.
4. Consider client and business context.
5. Approve, modify, or reject the recommendation.

### No-go list

The system must NOT:

- Automatically publish or delete content.
- Automatically change URLs or redirects.
- Automatically modify client websites.
- Automatically make irreversible SEO changes.
- Treat the priority score as a guaranteed prediction.
- Replace human judgment for final content decisions.

The model and action queue provide decision support only.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4) Monitoring and retrain triggers

### What to monitor

The action queue should be reviewed periodically for:

- Changes in model performance.
- Changes in the distribution of priority scores.
- Changes in the types of recommended actions.
- Increase in false positives or false negatives when new labels become available.
- Changes in the underlying traffic and content patterns.

### Retrain / review triggers

The model should be reviewed or retrained when:

1. Measured performance drops materially on newly labeled data.
2. The distribution of important input features changes substantially.
3. New content or client patterns are not represented in the training data.
4. The relationship between model signals and observed decline changes over time.

Retraining should be based on observed evidence rather than a fixed assumption that the model will remain valid indefinitely.

### Monitoring principle

Monitoring is intended to identify when the model's decision-support value may have changed. It does not guarantee future model performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [18]:
# Section 5 — Export ranked queue for the paper

from pathlib import Path
import json

OUTPUT_DIR = Path("/content/Flyrank-internship-/work/outputs")
FIGURES_DIR = Path("/content/Flyrank-internship-/work/figures")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Export the ranked queue
queue_path = OUTPUT_DIR / "ranked_action_queue.csv"
ranked_queue.to_csv(queue_path, index=False)

# Export summary metrics for traceability
summary = {
    "queue_size": int(len(ranked_queue)),
    "unique_clients": int(ranked_queue["client_id"].nunique()),
    "high_priority_items": int((ranked_queue["priority_score"] >= 3).sum()),
    "medium_priority_items": int(
        ((ranked_queue["priority_score"] >= 2) &
         (ranked_queue["priority_score"] < 3)).sum()
    ),
    "low_priority_items": int((ranked_queue["priority_score"] < 2).sum())
}

metrics_path = OUTPUT_DIR / "action_playbook_summary.json"

with open(metrics_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Exports created successfully.")
print(f"Queue: {queue_path}")
print(f"Summary: {metrics_path}")
print()
print(summary)

Exports created successfully.
Queue: /content/Flyrank-internship-/work/outputs/ranked_action_queue.csv
Summary: /content/Flyrank-internship-/work/outputs/action_playbook_summary.json

{'queue_size': 30000, 'unique_clients': 32, 'high_priority_items': 3637, 'medium_priority_items': 11306, 'low_priority_items': 15057}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.